In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
#data download and preprocessing

from datasets import load_dataset
from datasets import concatenate_datasets
from transformers import AutoTokenizer
from pympler import asizeof
from torch.utils.data import DataLoader
from collections import Counter

# Load datasets
mnli_dataset = load_dataset("glue", "mnli")
snli_dataset = load_dataset("snli")

# Remove 'idx' column from MNLI
mnli_dataset = {
    split: ds.remove_columns("idx") 
    for split, ds in mnli_dataset.items()
}

# Function to filter out invalid labels
def filter_valid(example):
    return example["label"] != -1

# Filter all SNLI splits - WITH MULTIPROCESSING
snli_dataset = {
    split: ds.filter(filter_valid, num_proc=4)  # Add num_proc
    for split, ds in snli_dataset.items()
}

# Filter all MNLI splits except test split - WITH MULTIPROCESSING
mnli_dataset = {
    split: ds.filter(filter_valid, num_proc=4) if "test" not in split else ds  # Add num_proc
    for split, ds in mnli_dataset.items()
}


train_dataset = concatenate_datasets([
    snli_dataset['train'], 
    mnli_dataset['train']
]).shuffle(seed=42)


validation_dataset = concatenate_datasets([
    snli_dataset['validation'], 
    mnli_dataset['validation_matched']
]).shuffle(seed=42)


validation_mismatched_dataset = mnli_dataset['validation_mismatched']


# Check the dataset structure
print(mnli_dataset)
print(snli_dataset)

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# tokenize_function with dynamic padding
def tokenize_function(example):
    return tokenizer(
        example["premise"],
        example["hypothesis"],
        padding=False,
        truncation=True,
        max_length=128
    )

# Tokenize each dataset separately - WITH MULTIPROCESSING
tokenized_train = train_dataset.map(tokenize_function, batched=True, num_proc=4)
tokenized_validation = validation_dataset.map(tokenize_function, batched=True, num_proc=4)
tokenized_validation_mismatched = validation_mismatched_dataset.map(tokenize_function, batched=True, num_proc=4)

# Tokenize test sets - WITH MULTIPROCESSING
tokenized_test_mnli_matched = mnli_dataset["test_matched"].map(tokenize_function, batched=True, num_proc=4)
tokenized_test_mnli_mismatched = mnli_dataset["test_mismatched"].map(tokenize_function, batched=True, num_proc=4)
tokenized_test_snli = snli_dataset["test"].map(tokenize_function, batched=True, num_proc=4)


# Change dataset format to use "labels" instead of "label"
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_validation = tokenized_validation.rename_column("label", "labels")
tokenized_validation_mismatched = tokenized_validation_mismatched.rename_column("label", "labels")

tokenized_test_mnli_matched = tokenized_test_mnli_matched.rename_column("label", "labels")
tokenized_test_mnli_mismatched = tokenized_test_mnli_mismatched.rename_column("label", "labels")
tokenized_test_snli = tokenized_test_snli.rename_column("label", "labels")

# Set format for PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_validation.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_validation_mismatched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])

tokenized_test_mnli_matched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_test_mnli_mismatched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_test_snli.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])


print(f"Train dataset size: {len(tokenized_train)}")
print(f"Validation dataset size: {len(tokenized_validation)}")
print(f"Validation mismatched size: {len(tokenized_validation_mismatched)}")

# attention_mask check 
print(tokenized_train[0]["attention_mask"])

# Check the train dataset dtype and size
size_in_bytes = asizeof.asizeof(tokenized_train)
size_in_mb = size_in_bytes / (1024 * 1024)
print(f"Total size of tokenized_train: {size_in_mb:.2f} MB")

# Check label distribution
train_labels = tokenized_train["labels"].tolist()
print(f"Label distribution in train: {Counter(train_labels)}")

print(tokenized_train[0]["input_ids"].shape, tokenized_train[0]["input_ids"].dtype)
print(tokenized_train[0]["token_type_ids"].shape, tokenized_train[0]["token_type_ids"].dtype)
print(tokenized_train[0]["attention_mask"].shape, tokenized_train[0]["attention_mask"].dtype)
print(tokenized_train[0]["labels"].shape, tokenized_train[0]["labels"].dtype)

# Check data types of each column
sample = tokenized_train[0]
for key, value in sample.items():
    print(f"Column: {key}, Data Type: {value.dtype}")

README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

mnli/test_matched-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

...

Map (num_proc=4):   0%|          | 0/9832 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9796 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9847 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9824 [00:00<?, ? examples/s]

Train dataset size: 942069
Validation dataset size: 19657
Validation mismatched size: 9832
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
Total size of tokenized_train: 1259.59 MB
Label distribution in train: Counter({0: 314315, 2: 314090, 1: 313664})
torch.Size([13]) torch.int64
torch.Size([13]) torch.int64
torch.Size([13]) torch.int64
torch.Size([]) torch.int64
Column: labels, Data Type: torch.int64
Column: input_ids, Data Type: torch.int64
Column: token_type_ids, Data Type: torch.int64
Column: attention_mask, Data Type: torch.int64


In [2]:
!pip install datasets transformers evaluate fvcore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.7 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=973f62a9bd242f171b83e21d350692cfb5ef3813b45f6e59535cc20681eb7864
  Stored in directory: /root/.cache/pip/wheels/65/71/95/3b8fde5c65c6e4a806e0867c1651dcc71a1cb2f3430e8f355f
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=f0e87117c2841e25285c8bbfc489254e7bb02835f3a5b0b285a5763cf3ad2a3e
  Stored in directory: /root/.cache/pip/wheels/ba/5e/16/6117f8fe7e9c0c161a795e10d94645ebcf301ccbd01f66d8ec
Successfully built fvcore iopath
  Attempting uninstall: fsspec
    Fou

In [20]:
import torch
import time
import numpy as np
from fvcore.nn import FlopCountAnalysis
from transformers import BertTokenizer

def benchmark_model(model, tokenizer, text1, text2, seq_len=128, use_mixed_precision=True):
    """
    Benchmarks a BERT-based model (e.g., BertForNLI or custom 23M model) using real tokenized inputs with a fixed sequence length.
    
    Args:
        model: BERT-based model (e.g., BertForNLI or custom model).
        tokenizer: Preloaded tokenizer (e.g., BertTokenizer).
        text1 (str): First input sentence.
        text2 (str): Second input sentence (for NLI pair).
        seq_len (int): Sequence length for tokenization (default: 128).
        use_mixed_precision (bool): Whether to use mixed precision (bf16/fp16) if on CUDA.
    
    Returns:
        List of throughput results for different batch sizes.
    """
    device = next(model.parameters()).device
    
    # Setup mixed precision
    if use_mixed_precision and device.type == "cuda":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        print(f"Using mixed precision: {dtype}")
    else:
        dtype = torch.float32
        print("Using full precision: float32")
    
    # Prepare real tokenized inputs
    inputs = tokenizer(
        text1,
        text2,
        max_length=seq_len,  # Fixed to 128
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = inputs['input_ids'].to(device)  # Shape: [1, seq_len]
    token_type_ids = inputs['token_type_ids'].to(device)  # Shape: [1, seq_len]
    attention_mask = inputs['attention_mask'].to(device)  # Shape: [1, seq_len]
    labels = torch.tensor([0], device=device)  # Example label (0 for contradiction)
    
    model.eval()
    
    print("="*70)
    print("MODEL STATISTICS")
    print("="*70)
    
    # Parameter Count
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    
    # FLOPs Calculation
    print("\n" + "="*70)
    print("FLOPS ANALYSIS")
    print("="*70)
    
    with torch.no_grad():
        # Check if model is Hugging Face BERT or custom
        try:
            # For Hugging Face BertForSequenceClassification
            flops = FlopCountAnalysis(model, (input_ids, attention_mask, token_type_ids, None))
        except Exception as e:
            print(f"FLOPs analysis failed: {e}. Falling back to input_ids only.")
            flops = FlopCountAnalysis(model, (input_ids,))  # For custom models
        total_flops = flops.total()
        
        print(f"Total FLOPs per forward pass: {total_flops/1e9:.2f} GFLOPs")
        print(f"FLOPs per sample: {total_flops/1e9:.2f} GFLOPs")
        print(f"Total FLOPs for sequence: {total_flops/1e12:.4f} TFLOPs")
    
    # Memory Usage
    print("\n" + "="*70)
    print("MEMORY USAGE")
    print("="*70)
    
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        if use_mixed_precision and device.type == "cuda":
            with torch.autocast(device_type='cuda', dtype=dtype):
                _ = model(input_ids, attention_mask, token_type_ids, labels)
        else:
            _ = model(input_ids, attention_mask, token_type_ids, labels)
    
    memory_allocated = torch.cuda.max_memory_allocated() / 1e6
    print(f"Peak GPU memory (batch_size=1): {memory_allocated:.2f} MB")
    
    # Latency Measurement
    print("\n" + "="*70)
    print("LATENCY MEASUREMENT (batch_size=1)")
    print("="*70)
    
    # Warmup
    print("Warming up...")
    with torch.no_grad():
        for _ in range(20):
            if use_mixed_precision and device.type == "cuda":
                with torch.autocast(device_type='cuda', dtype=dtype):
                    _ = model(input_ids, attention_mask, token_type_ids, labels)
            else:
                _ = model(input_ids, attention_mask, token_type_ids, labels)
    torch.cuda.synchronize()
    
    # Actual measurement
    print("Measuring latency...")
    latencies = []
    num_iterations = 100
    
    with torch.no_grad():
        for _ in range(num_iterations):
            torch.cuda.synchronize()
            start = time.perf_counter()
            
            if use_mixed_precision and device.type == "cuda":
                with torch.autocast(device_type='cuda', dtype=dtype):
                    _ = model(input_ids, attention_mask, token_type_ids, labels)
            else:
                _ = model(input_ids, attention_mask, token_type_ids, labels)
            
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - start) * 1000)  # ms
    
    latencies = np.array(latencies)
    mean_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    print(f"Latency: {mean_latency:.2f} ± {std_latency:.2f} ms")
    
    # Throughput Measurement
    print("\n" + "="*70)
    print("THROUGHPUT MEASUREMENT")
    print("="*70)
    
    batch_sizes = [1, 8, 16, 32]
    throughput_results = []
    
    for batch_size in batch_sizes:
        try:
            # Repeat tokenized inputs for larger batch sizes
            input_ids_batch = input_ids.repeat(batch_size, 1)
            token_type_ids_batch = token_type_ids.repeat(batch_size, 1)
            attention_mask_batch = attention_mask.repeat(batch_size, 1)
            labels_batch = labels.repeat(batch_size)
            
            # Warmup
            with torch.no_grad():
                for _ in range(10):
                    if use_mixed_precision and device.type == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, attention_mask_batch, token_type_ids_batch, labels_batch)
                    else:
                        _ = model(input_ids_batch, attention_mask_batch, token_type_ids_batch, labels_batch)
            torch.cuda.synchronize()
            
            # Measure
            num_iterations = 50
            torch.cuda.synchronize()
            start = time.perf_counter()
            
            with torch.no_grad():
                for _ in range(num_iterations):
                    if use_mixed_precision and device.type == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, attention_mask_batch, token_type_ids_batch, labels_batch)
                    else:
                        _ = model(input_ids_batch, attention_mask_batch, token_type_ids_batch, labels_batch)
            
            torch.cuda.synchronize()
            elapsed_time = time.perf_counter() - start
            
            samples_per_sec = (batch_size * num_iterations) / elapsed_time
            tokens_per_sec = samples_per_sec * seq_len  # Use fixed seq_len=128
            
            # Memory check
            torch.cuda.reset_peak_memory_stats()
            with torch.no_grad():
                if use_mixed_precision and device.type == "cuda":
                    with torch.autocast(device_type='cuda', dtype=dtype):
                        _ = model(input_ids_batch, attention_mask_batch, token_type_ids_batch, labels_batch)
                else:
                    _ = model(input_ids_batch, attention_mask_batch, token_type_ids_batch, labels_batch)
            memory_used = torch.cuda.max_memory_allocated() / 1e9
            
            throughput_results.append({
                'batch_size': batch_size,
                'samples_per_sec': samples_per_sec,
                'tokens_per_sec': tokens_per_sec,
                'memory_gb': memory_used
            })
            
            print(f"Batch size {batch_size:3d}: {samples_per_sec:7.2f} samples/sec | "
                  f"{tokens_per_sec:10.2f} tokens/sec | Memory: {memory_used:.2f} GB")
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"Batch size {batch_size:3d}: OOM (Out of Memory)")
                torch.cuda.empty_cache()
                break
            else:
                raise e
    
    # Performance Summary
    print("\n" + "="*70)
    print("PERFORMANCE SUMMARY")
    print("="*70)
    
    if throughput_results:
        max_throughput = max(throughput_results, key=lambda x: x['tokens_per_sec'])
        print(f"Peak throughput: {max_throughput['tokens_per_sec']:.2f} tokens/sec "
              f"(batch_size={max_throughput['batch_size']})")
        print(f"Single sample latency: {mean_latency:.2f} ± {std_latency:.2f} ms")
        print(f"FLOPs per forward pass: {total_flops/1e9:.2f} GFLOPs")
        
        # Theoretical achieved FLOPS
        mean_latency_sec = mean_latency / 1000
        theoretical_flops = total_flops / mean_latency_sec / 1e12
        print(f"Achieved compute: {theoretical_flops:.2f} TFLOPS")
    
    return throughput_results




In [22]:
import torch
import torch.nn as nn
import inspect
from transformers import BertConfig, BertForSequenceClassification


# BERT-45M
class BertForNLI(nn.Module):
    def __init__(self, vocab_size=30522, num_labels=3):
        super().__init__()
        
        # === Custom BERT config (~23M params) ===
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=512,
            num_hidden_layers=9,
            num_attention_heads=8,
            intermediate_size=2048,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1,
            max_position_embeddings=512,
            type_vocab_size=2,
            num_labels=num_labels   # ✅ important for classification
        )
        
        # Classification head instead of MLM
        self.model = BertForSequenceClassification(config)
        

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        """
        Forward pass for NLI (sequence classification).
        Returns logits + loss (if labels are provided).
        """
        outputs = self.model(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs.logits, outputs.loss if labels is not None else None

    def configure_optimizers(self, weight_decay, learning_rate, device="cuda", verbose=False):
        """
        AdamW optimizer with weight decay only on weight matrices.
        """
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}

        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

        optim_groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ]

        fused_available = "fused" in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and "cuda" in device

        if verbose:
            num_decay = sum(p.numel() for p in decay_params)
            num_nodecay = sum(p.numel() for p in nodecay_params)
            print(f"decay params: {len(decay_params)} tensors, {num_decay/1e6:.2f}M params")
            print(f"nodecay params: {len(nodecay_params)} tensors, {num_nodecay/1e6:.2f}M params")
            print(f"Using fused AdamW: {use_fused}")

        optimizer = torch.optim.AdamW(
            optim_groups,
            lr=learning_rate,
            betas=(0.9, 0.999),
            eps=1e-8,
            fused=use_fused
        )
        return optimizer


# Define input texts
text1 = "The man is walking down the street."
text2 = "A person is outside."

# 45M model
model_45m = BertForNLI(num_labels=3).to(device)
print("\nBenchmarking 45M model...")
results_45m = benchmark_model(model_45m, tokenizer, text1, text2, seq_len=128, use_mixed_precision=True)


Benchmarking 45M model...
Using mixed precision: torch.bfloat16
MODEL STATISTICS
Total parameters: 44,527,107 (44.53M)
Trainable parameters: 44,527,107 (44.53M)

FLOPS ANALYSIS
Total FLOPs per forward pass: 3.63 GFLOPs
FLOPs per sample: 3.63 GFLOPs
Total FLOPs for sequence: 0.0036 TFLOPs

MEMORY USAGE
Peak GPU memory (batch_size=1): 964.25 MB

LATENCY MEASUREMENT (batch_size=1)
Warming up...
Measuring latency...
Latency: 10.56 ± 0.15 ms

THROUGHPUT MEASUREMENT
Batch size   1:   95.91 samples/sec |   12276.26 tokens/sec | Memory: 0.96 GB
Batch size   8:  427.70 samples/sec |   54745.67 tokens/sec | Memory: 0.98 GB
Batch size  16:  471.37 samples/sec |   60335.66 tokens/sec | Memory: 1.01 GB
Batch size  32:  531.54 samples/sec |   68037.47 tokens/sec | Memory: 1.07 GB

PERFORMANCE SUMMARY
Peak throughput: 68037.47 tokens/sec (batch_size=32)
Single sample latency: 10.56 ± 0.15 ms
FLOPs per forward pass: 3.63 GFLOPs
Achieved compute: 0.34 TFLOPS


In [23]:
# BERT-23M
class BertForNLI(nn.Module):
    def __init__(self, vocab_size=30522, num_labels=3):
        super().__init__()
        
        # === Custom BERT config (~23M params) ===
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=384,
            num_hidden_layers=6,
            num_attention_heads=6,
            intermediate_size=1536,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1,
            max_position_embeddings=512,
            type_vocab_size=2,
            num_labels=num_labels   # ✅ important for classification
        )
        
        # Classification head instead of MLM
        self.model = BertForSequenceClassification(config)
        

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        """
        Forward pass for NLI (sequence classification).
        Returns logits + loss (if labels are provided).
        """
        outputs = self.model(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs.logits, outputs.loss if labels is not None else None

    def configure_optimizers(self, weight_decay, learning_rate, device="cuda", verbose=False):
        """
        AdamW optimizer with weight decay only on weight matrices.
        """
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}

        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

        optim_groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ]

        fused_available = "fused" in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and "cuda" in device

        if verbose:
            num_decay = sum(p.numel() for p in decay_params)
            num_nodecay = sum(p.numel() for p in nodecay_params)
            print(f"decay params: {len(decay_params)} tensors, {num_decay/1e6:.2f}M params")
            print(f"nodecay params: {len(nodecay_params)} tensors, {num_nodecay/1e6:.2f}M params")
            print(f"Using fused AdamW: {use_fused}")

        optimizer = torch.optim.AdamW(
            optim_groups,
            lr=learning_rate,
            betas=(0.9, 0.999),
            eps=1e-8,
            fused=use_fused
        )
        return optimizer


# 45M model
model_23m = BertForNLI(num_labels=3).to(device)
print("\nBenchmarking 23M model...")
results_45m = benchmark_model(model_23m, tokenizer, text1, text2, seq_len=128, use_mixed_precision=True)


Benchmarking 23M model...
Using mixed precision: torch.bfloat16
MODEL STATISTICS
Total parameters: 22,714,371 (22.71M)
Trainable parameters: 22,714,371 (22.71M)

FLOPS ANALYSIS
Total FLOPs per forward pass: 1.36 GFLOPs
FLOPs per sample: 1.36 GFLOPs
Total FLOPs for sequence: 0.0014 TFLOPs

MEMORY USAGE
Peak GPU memory (batch_size=1): 929.30 MB

LATENCY MEASUREMENT (batch_size=1)
Warming up...
Measuring latency...
Latency: 7.46 ± 0.83 ms

THROUGHPUT MEASUREMENT
Batch size   1:  139.71 samples/sec |   17882.46 tokens/sec | Memory: 0.93 GB
Batch size   8:  941.23 samples/sec |  120476.92 tokens/sec | Memory: 0.95 GB
Batch size  16: 1149.30 samples/sec |  147110.34 tokens/sec | Memory: 0.97 GB
Batch size  32: 1260.71 samples/sec |  161371.29 tokens/sec | Memory: 1.01 GB

PERFORMANCE SUMMARY
Peak throughput: 161371.29 tokens/sec (batch_size=32)
Single sample latency: 7.46 ± 0.83 ms
FLOPs per forward pass: 1.36 GFLOPs
Achieved compute: 0.18 TFLOPS
